Malformed Product Label - https://drive.google.com/file/d/1CrjKNdexd-MYhsYC9bNWL3y8xERr-lQT/view?usp=drive_link

In [ ]:
ALTER STAGE AGENT_COMMERCE.PRODUCTS.PRODUCT_LABELS_STAGE REFRESH;
ls @AGENT_COMMERCE.PRODUCTS.PRODUCT_LABELS_STAGE/ANA-Lipstick-Label.jpeg;


Misformed Label Topics Extract - Using AI SQL

In [ ]:
WITH raw_extraction AS (
    SELECT AI_COMPLETE(
        'claude-opus-4-6',
        'Extract ALL information from this cosmetic Drug Facts label. Return ONLY a JSON object:
{
  "brand": "",
  "product_name": "",
  "net_weight": "",
  "spf": "",
  "warnings": [],
  "directions": [],
  "other_information": [],
  "inactive_ingredients": []
}
Include ALL text from Warnings, Directions, and Other Information sections.
Read ONLY what is printed. Do not add ingredients not visible on label.',
        TO_FILE('@AGENT_COMMERCE.PRODUCTS.PRODUCT_LABELS_STAGE', 'ANA-Lipstick-Label.jpeg')
    ) AS raw_result
),
parsed AS (
    SELECT TRY_PARSE_JSON(
        REGEXP_SUBSTR(raw_result, '\\{.*\\}', 1, 1, 's')
    ) AS data
    FROM raw_extraction
)
-- Product Info
SELECT 1 AS sort, '🏷️ Brand' AS "Property", data:brand::VARCHAR AS "Value" FROM parsed
UNION ALL SELECT 2, '📦 Product Name', data:product_name::VARCHAR FROM parsed
UNION ALL SELECT 3, '⚖️ Net Weight', data:net_weight::VARCHAR FROM parsed
UNION ALL SELECT 4, '☀️ SPF', COALESCE(data:spf::VARCHAR, 'N/A') FROM parsed
UNION ALL SELECT 5, '─────────────', '─────────────────────────────────────────' FROM parsed

-- Warnings Section
UNION ALL SELECT 6, '⚠️ WARNINGS', '(' || ARRAY_SIZE(data:warnings) || ' items)' FROM parsed
UNION ALL SELECT 10 + w.index, '  ⚠️ #' || (w.index + 1), w.value::VARCHAR 
FROM parsed, LATERAL FLATTEN(input => data:warnings, OUTER => TRUE) w WHERE w.value IS NOT NULL

-- Directions Section
UNION ALL SELECT 50, '─────────────', '─────────────────────────────────────────' FROM parsed
UNION ALL SELECT 51, '📝 DIRECTIONS', '(' || ARRAY_SIZE(data:directions) || ' items)' FROM parsed
UNION ALL SELECT 60 + d.index, '  📝 #' || (d.index + 1), d.value::VARCHAR 
FROM parsed, LATERAL FLATTEN(input => data:directions, OUTER => TRUE) d WHERE d.value IS NOT NULL

-- Other Information Section
UNION ALL SELECT 80, '─────────────', '─────────────────────────────────────────' FROM parsed
UNION ALL SELECT 81, 'ℹ️ OTHER INFO', '(' || ARRAY_SIZE(data:other_information) || ' items)' FROM parsed
UNION ALL SELECT 90 + o.index, '  ℹ️ #' || (o.index + 1), o.value::VARCHAR 
FROM parsed, LATERAL FLATTEN(input => data:other_information, OUTER => TRUE) o WHERE o.value IS NOT NULL

-- Separator
UNION ALL SELECT 100, '─────────────', '─────────────────────────────────────────' FROM parsed

-- Ingredients Section
UNION ALL SELECT 101, '🧪 INGREDIENTS', '(' || ARRAY_SIZE(data:inactive_ingredients) || ' items)' FROM parsed
UNION ALL SELECT 110 + i.index, '  🧪 #' || (i.index + 1), i.value::VARCHAR 
FROM parsed, LATERAL FLATTEN(input => data:inactive_ingredients, OUTER => TRUE) i WHERE i.value IS NOT NULL

ORDER BY sort;

## Product Label Extraction Pipeline

**Architecture**:  
Stage Images → File Registry → Dynamic Table (AI_COMPLETE) → Cortex Search → Cortex Data Agent (agent commerce agent)

| Component | Object | Refresh |
|-----------|--------|---------|
| AI Extraction | `PRODUCT_LABEL_EXTRACT` (Dynamic Table) | Every 5 minutes |
| Search | `PRODUCT_LABEL_SEARCH` (Cortex Search) | Every 1 hour |

### File Registry Table - References Unstructured Product Labels in your object storage

In [ ]:
USE ROLE AGENT_COMMERCE_ROLE;
USE WAREHOUSE AGENT_COMMERCE_WH;
USE SCHEMA AGENT_COMMERCE.PRODUCTS;

CREATE OR REPLACE TABLE AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_FILES (
    FILE_PATH VARCHAR
);

INSERT INTO AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_FILES (FILE_PATH)
SELECT RELATIVE_PATH
FROM DIRECTORY(@AGENT_COMMERCE.PRODUCTS.PRODUCT_LABELS_STAGE);

### Enriched Product Catalog with Product Label Images (Unstructured)
#### Self Refereshing Unstructued Pipeline - Combine Dynamic Table + AI SQL 

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_EXTRACT
    TARGET_LAG = '5 minutes'
    WAREHOUSE = AGENT_COMMERCE_WH
    REFRESH_MODE = FULL
AS
WITH raw_extraction AS (
    SELECT
        f.FILE_PATH,
        SPLIT_PART(REPLACE(REPLACE(f.FILE_PATH, '.png', ''), '.jpeg', ''), '_', 1) AS SKU,
        AI_COMPLETE(
            'claude-opus-4-6',
            'Extract ALL information from this cosmetic Drug Facts label. Return ONLY a JSON object:
{
  "brand": "",
  "product_name": "",
  "net_weight": "",
  "spf": "",
  "warnings": [],
  "directions": [],
  "other_information": [],
  "inactive_ingredients": []
}
Include ALL text from Warnings, Directions, and Other Information sections.
Read ONLY what is printed. Do not add ingredients not visible on label.',
            TO_FILE('@AGENT_COMMERCE.PRODUCTS.PRODUCT_LABELS_STAGE', f.FILE_PATH)
        ) AS RAW_RESULT
    FROM AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_FILES f
),
parsed AS (
    SELECT
        FILE_PATH,
        SKU,
        RAW_RESULT,
        TRY_PARSE_JSON(REGEXP_SUBSTR(RAW_RESULT, '\\{.*\\}', 1, 1, 's')) AS DATA
    FROM raw_extraction
)
SELECT
    FILE_PATH,
    SKU,
    DATA:brand::VARCHAR AS BRAND,
    DATA:product_name::VARCHAR AS PRODUCT_NAME,
    DATA:net_weight::VARCHAR AS NET_WEIGHT,
    DATA:spf::VARCHAR AS SPF,
    ARRAY_TO_STRING(DATA:warnings, ' | ') AS WARNINGS,
    ARRAY_TO_STRING(DATA:directions, ' | ') AS DIRECTIONS,
    ARRAY_TO_STRING(DATA:other_information, ' | ') AS OTHER_INFORMATION,
    ARRAY_TO_STRING(DATA:inactive_ingredients, ' | ') AS INACTIVE_INGREDIENTS,
    DATA AS RAW_JSON
FROM parsed;

### Integrate Product Label (Unstructured) to Agentic Commerce Data Agent
#### Self Refereshing Agent: Cortex Search + Dynaminc Table + AISQL 

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_SEARCH_V2
    ON SEARCH_CONTENT
    ATTRIBUTES BRAND, PRODUCT_NAME, SKU, FILE_PATH
    WAREHOUSE = AGENT_COMMERCE_WH
    TARGET_LAG = '1 hour'
AS (
    SELECT
        SKU,
        FILE_PATH,
        BRAND,
        PRODUCT_NAME,
        NET_WEIGHT,
        SPF,
        COALESCE(BRAND, '') || ' ' ||
        COALESCE(PRODUCT_NAME, '') || ' ' ||
        COALESCE(WARNINGS, '') || ' ' ||
        COALESCE(DIRECTIONS, '') || ' ' ||
        COALESCE(OTHER_INFORMATION, '') || ' ' ||
        COALESCE(INACTIVE_INGREDIENTS, '') AS SEARCH_CONTENT
    FROM AGENT_COMMERCE.PRODUCTS.PRODUCT_LABEL_EXTRACT
);